# Live lab 1 · One question, one real LLM call

Read after Chapter 6 and Appendix A.5–A.8. Goal: identify exactly what enters the model and what comes back. You do not need RAG or agents yet.

**Live execution.** Restarting and running this notebook performs new local model inference. Saved outputs are one actual development run, not guaranteed answers. No paid API is used. First follow [Python setup](https://chikuang.github.io/teaching/genai/python-workflow.html), then run these commands in a terminal at the cloned course root:

```bash
python -m pip install -r requirements.txt -r requirements-llm.txt
python labs/download_model.py
python -m jupyterlab
```

Open the notebook in JupyterLab. Press **Shift+Enter** in order. Model weights take about 3 GB on disk; CPU float32 inference needs several additional GB of RAM. The first model-loading cell can take time. Do not run it repeatedly while it is busy. Inference remains on your machine.


## 1. Find the course and its Python code

`Path` represents a folder, `/` joins names, and `sys.path` tells Python where local modules can be imported. The search checks the current folder and its parents, so this works from either the repository root or `labs/notebooks/`. These operations load no model and generate no answers.

In [ ]:
from pathlib import Path
import os
import sys
import json

start = Path(os.environ.get('GENAI_COURSE_ROOT', Path.cwd())).resolve()
root = next((p for p in [start, *start.parents]
             if (p / 'labs/model.json').is_file()), None)
if root is None:
    raise FileNotFoundError('Open this notebook inside the cloned Teaching-GenAI folder')
sys.path.insert(0, str(root))
from labs.llm_lab import LocalLM
from labs.notebook_support import save_run
print('Course folder:', root.name)
print('Python:', sys.version.split()[0])

## 2. Load the fixed tokenizer and weights

`LocalLM` is a small wrapper in `labs/llm_lab.py`. Its constructor checks the downloaded model identity, loads `AutoTokenizer` and `AutoModelForCausalLM`, sets CPU float32 and evaluation mode, and creates an empty call log. `lm.tokenizer` and `lm.model` expose those two objects. Loading fitted parameters is not training.

In [ ]:
lm = LocalLM(root / '_models/qwen2.5-1.5b')
print(lm.versions)
print('Model loaded; no question has been generated yet.')

## 3. Construct a single request

A list holds messages in order. Each dictionary has a speaker (`role`) and text (`content`). Change only the user text on your second experiment, so you can identify what changed.

In [ ]:
messages = [
    {'role': 'system', 'content': 'Explain statistical ideas in one short sentence.'},
    {'role': 'user', 'content': 'What is Monte Carlo error?'},
]
print(json.dumps(messages, indent=2))

## 4. Inspect the actual model input

The chat template adds the model-specific role delimiters. `tokenize=False` first returns a string; the next call turns it into integer IDs. `add_special_tokens=False` avoids adding duplicate control tokens. `return_tensors="pt"` asks for PyTorch tensors. Predict: the first shape dimension is one because we have one prompt, not one because we have one token.

In [ ]:
text = lm.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True)
inputs = lm.tokenizer(text, return_tensors='pt', add_special_tokens=False)
print(text)
print('Token tensor shape:', tuple(inputs.input_ids.shape))
print('First twelve IDs:', inputs.input_ids[0, :12].tolist())
assert inputs.input_ids.ndim == 2 and inputs.input_ids.shape[0] == 1

## 5. Generate once

Calling `lm(messages)` runs `model.generate(**inputs, do_sample=False, ...)` under `torch.inference_mode()`, slices off the input prefix, decodes only the new IDs, and logs the raw answer. `**inputs` means pass each dictionary item as a named argument. Greedy decoding chooses a largest-probability next token; it does not certify truth. `max_new_tokens=64` is a length budget, not a sample size for statistical inference.

In [ ]:
answer = lm(messages, max_new_tokens=64)
print('Raw model answer:', answer)
call = lm.calls[-1]
print('Prompt tokens:', call['prompt_tokens'])
print('Generated tokens:', call['generated_tokens'])
print('Stopped because:', call['stop_reason'])
assert len(lm.calls) == 1
assert call['prompt_tokens'] == inputs.input_ids.shape[1]
assert 0 < call['generated_tokens'] <= 64

## 6. Check meaning, then save

Technical acceptance: one real call, bounded output, recorded runtime and stop reason. Scientific acceptance is separate: does the answer say that Monte Carlo error arises from a finite number of simulated draws? Does it avoid confusing it with model misspecification or parameter-estimation error? A nonempty string alone does not pass this rubric. If truncated, record that failure before trying a larger budget as a new condition.

In [ ]:
record_path = save_run(root, 'first-llm', lm, {'answer': answer, 'semantic_review': 'Reader checks the stated Monte Carlo rubric'})

## Practice

1. **Predict:** which objects change if you alter the prompt but keep the weights fixed?
2. **Implement:** replace the user question with one about estimation error; run steps 3–6 and compare both raw answers. The `len(lm.calls)==1` assertion applies to a fresh first run; after adding calls, check the expected new count instead.
3. **Investigate:** predeclare two related prompts and a semantic rubric. Report failures and token counts. Two authored questions cannot establish a population accuracy.

Continue to Chapter 10, then [live lab 2](02-rag-bridge.ipynb).